# Sign-off Journey

Follows the same lines through three sign-offs, so a number that looks wrong in the
latest one can be traced back to the month it moved.

Four tabs - **TF 7599 DSP Rule** and the three **Experian DSP $75 Rule** bands - out of
the September, October and November workbooks. Line names come from column **G, rows
9-35**; the figures from **H, I and J**, years 1 to 3. Nothing to the right of J is read.

**What moved** sits at the top: the largest changes into the latest sign-off, ranked.
Underneath, one table per tab with the lines down the page and each year reading left to
right - a month, what changed, the next month.

A change is measured against the **larger** of the two figures, not the earlier one, so a
line creeping up off a near-zero base cannot outrank one that halved, and a sign change is
not infinite. Change cells are inked by how far the line moved: the colour says how big,
never whether it is good, because what counts as good differs line by line. Hover one for
the move in full.

Out comes `signoff_journey.html` to open or mail, `signoff_journey_artifact.html` to
publish, and `signoff_journey.xlsx` - a sheet per tab laid out the same way, plus a
Movers sheet you can filter.

Run the cells in order. The only cell you edit is the first one.

In [ ]:
# ── 1. Settings ───────────────────────────────────────────────────────────────
import datetime, math, os
import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import column_index_from_string, get_column_letter

PROJECT = "/Users/evernjoshua/Madam Claude Automation"
os.chdir(PROJECT)

# One workbook per sign-off, in the order they should read across the page.
# Edit the file names to match what is actually on disk.
BOOKS = [
    ("Sep-2026", "09'26 P&L Forecast - All.xlsm"),
    ("Oct-2026", "10'26 P&L Forecast - All.xlsm"),
    ("Nov-2026", "11'26 P&L Forecast - All.xlsm"),
]

# The tabs to pull out of every workbook, in the order they should stack.
SHEETS = [
    "TF 7599 DSP Rule",
    "Experian DSP $75 Rule <=600",
    "Experian DSP $75 Rule 601-639",
    "Experian DSP $75 Rule 640+",
]

# Column G carries the line names, rows 9 to 35, and H, I and J carry years 1,
# 2 and 3 beside them. That is the same layout the heat map reads: its unit C/O
# comes from H11/I11/J11, which sits against "Cumulative Unit C/O Rate (%)" at
# G11, and its ROA from H22/I22/J22 against "Cumulative ROA - Annualized" at G22.
LABEL_COLUMN = "G"
FIRST_ROW, LAST_ROW = 9, 35

# The block is G to J and STOPS there: names in G, years 1 to 3 in H, I and J.
# Nothing to the right of J is read, whatever else the sheet holds - cell 3 drops
# any column named below that falls outside this range and says so.
FIRST_VALUE_COLUMN, LAST_VALUE_COLUMN = "H", "J"

# Which of those columns to show, and what to call each. Comment a line out and
# that year leaves the table.
YEARS = {
    "Yr 1": "H",
    "Yr 2": "I",
    "Yr 3": "J",
}

# How much of a move counts as "big" when the change columns are shaded: a line
# that moved this much against the prior sign-off, relatively, is fully inked.
BIG_MOVE = 0.25

# A line with a name but no figures in any month is read as a section heading -
# INCOME STATEMENT, CASH FLOW - and groups the lines beneath it. Everything
# before the first heading sits under this name.
FIRST_SECTION = "Portfolio"

# These four were renamed: the $75 / 7599 names are current, the $49 / 4999 ones
# are what the same tabs were called before. An older sign-off still carries the
# old name, so each tab is looked up under its current name first and then under
# every former name here, newest first. The row keeps the current name whichever
# one matched, and cell 3 says which workbook used which - a rename partway
# through a series is exactly what quietly breaks a comparison.
SHEET_ALIASES = {
    "TF 7599 DSP Rule":              ["TF 4999 DSP Rule"],
    "Experian DSP $75 Rule <=600":   ["Experian DSP $49 Rule <=600"],
    "Experian DSP $75 Rule 601-639": ["Experian DSP $49 Rule 601-639"],
    "Experian DSP $75 Rule 640+":    ["Experian DSP $49 Rule 640+"],
}

# How each line is written. The first pattern that appears in the name wins;
# anything unmatched falls back to LAST. Edit freely - these are only display.
NUMBER_STYLES = [
    ("rate (%)",  dict(fmt="0.00%",         py="{:.2%}")),
    ("roa",       dict(fmt="0.00%",         py="{:.2%}")),
    ("(%)",       dict(fmt="0.00%",         py="{:.2%}")),
    ("accounts",  dict(fmt="#,##0",         py="{:,.0f}")),
    ("/accts",    dict(fmt='"$"#,##0.00',   py="${:,.2f}")),
    ("/avg",      dict(fmt='"$"#,##0.00',   py="${:,.2f}")),
    ("per ",      dict(fmt='"$"#,##0.00',   py="${:,.2f}")),
]
LAST = dict(fmt='"$"#,##0', py="${:,.0f}")

OUTPUT_XLSX = "signoff_journey.xlsx"
OUTPUT_HTML = "signoff_journey.html"

STAMP = datetime.datetime.now().strftime("%d %B %Y, %H:%M")

print(f"settings loaded - {len(BOOKS)} sign-offs, {len(SHEETS)} tabs, "
      f"names {LABEL_COLUMN}{FIRST_ROW}:{LABEL_COLUMN}{LAST_ROW}, "
      f"figures {FIRST_VALUE_COLUMN}-{LAST_VALUE_COLUMN}, "
      f"{len(YEARS)} period(s) shown")
for label, path in BOOKS:
    print(f"  {label:<9} {'FOUND    ' if os.path.exists(path) else 'NOT FOUND'}  {path}")

## 2. What the workbooks hold

Cell 3 prints the line items it found, grouped by section, and names any tab it could not
find along with the tabs that workbook does have. Copy a spelling out of that list into
`SHEETS` or `SHEET_ALIASES` if one does not match.

In [ ]:
# ── 3. Read the three workbooks ───────────────────────────────────────────────
NAN = float("nan")

def _squash(s):
    return "".join(str(s).split()).lower()

def _num(v):
    # cached Excel values arrive as numbers, strings, None or errors
    if v is None or isinstance(v, bool):
        return NAN
    if isinstance(v, (int, float)):
        return float(v)
    t = str(v).strip()
    if not t or t.startswith("#"):
        return NAN
    neg = t.startswith("(") and t.endswith(")")
    t = t.strip("()").replace(",", "").replace("$", "").replace(" ", "")
    pct = t.endswith("%")
    t = t.rstrip("%")
    try:
        x = float(t)
    except ValueError:
        return NAN
    if pct:
        x /= 100.0
    return -x if neg else x

def fmt_value(x, py):
    # the sign belongs outside the currency symbol: -$354,514, not $-354,514
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    body = py.format(abs(x))
    return ("-" + body) if x < 0 else body

def style_for(label):
    low = str(label).lower()
    for needle, style in NUMBER_STYLES:
        if needle in low:
            return style
    return LAST

def names_for(name):
    # the current name first, then any former ones
    also = SHEET_ALIASES.get(name, [])
    if isinstance(also, str):
        also = [also]
    return [name] + list(also)

def find_sheet(wb, name):
    # (tab as the workbook spells it, the name it answered to) or (None, None)
    have = {_squash(a): a for a in wb.sheetnames}
    for candidate in names_for(name):
        hit = have.get(_squash(candidate))
        if hit:
            return hit, candidate
    return None, None

# Nothing outside H:L is ever read. A column named in YEARS that falls outside
# it is dropped here rather than quietly reaching into a neighbouring block.
_LO = column_index_from_string(FIRST_VALUE_COLUMN)
_HI = column_index_from_string(LAST_VALUE_COLUMN)
_OUT = {y: c for y, c in YEARS.items()
        if not (_LO <= column_index_from_string(c) <= _HI)}
if _OUT:
    print(f"outside {FIRST_VALUE_COLUMN}:{LAST_VALUE_COLUMN}, so not read: {_OUT}")
YEARS = {y: c for y, c in YEARS.items() if y not in _OUT}
if not YEARS:
    raise SystemExit(f"no value column left inside "
                     f"{FIRST_VALUE_COLUMN}:{LAST_VALUE_COLUMN}")

def read_book(path):
    # {tab: {row number: (label, {year: value})}} for one sign-off
    wb = load_workbook(path, data_only=True)
    try:
        out, missing = {}, []
        for name in SHEETS:
            actual, answered_to = find_sheet(wb, name)
            if actual is None:
                missing.append(name)
                continue
            if _squash(answered_to) != _squash(name):
                RENAMES.append((label_of(path), name, actual))
            ws = wb[actual]
            rows = {}
            for r in range(FIRST_ROW, LAST_ROW + 1):
                label = ws[f"{LABEL_COLUMN}{r}"].value
                label = " ".join(str(label).split()) if label is not None else ""
                rows[r] = (label, {y: _num(ws[f"{c}{r}"].value)
                                   for y, c in YEARS.items()})
            out[name] = rows
        if missing:
            print(f"    tabs NOT FOUND in {os.path.basename(path)}: {missing}")
            print(f"    tabs present: {wb.sheetnames}")
    finally:
        wb.close()
    return out

RENAMES = []            # [(month, current name, the name that tab actually has)]

def label_of(path):
    return next((m for m, p in BOOKS if p == path), os.path.basename(path))

BOOK_DATA, MONTHS = {}, []
for label, path in BOOKS:
    if not os.path.exists(path):
        print(f"skipping {label}: no file at {path!r}")
        continue
    BOOK_DATA[label] = read_book(path)
    MONTHS.append(label)
    print(f"{label}: {len(BOOK_DATA[label])} of {len(SHEETS)} tabs read from "
          f"{os.path.basename(path)}")

if not MONTHS:
    raise SystemExit("no workbooks could be read - check the file names in cell 1")

if RENAMES:
    print("\nmatched under a former name:")
    for month, current, actual in RENAMES:
        print(f"  {month}: {current!r} found as {actual!r}")

# ---- the line names, taken from the first month that has them ---------------
# A row with a name but no figures anywhere is a heading; a row with no name at
# all is a spacer and is dropped.
LINES = []            # [(row, section, label, style)]
section = FIRST_SECTION
for r in range(FIRST_ROW, LAST_ROW + 1):
    label = ""
    for m in MONTHS:
        for tab in BOOK_DATA[m].values():
            if tab.get(r) and tab[r][0]:
                label = tab[r][0]
                break
        if label:
            break
    if not label:
        continue                                   # spacer row
    has_value = any(not math.isnan(v)
                    for m in MONTHS
                    for tab in BOOK_DATA[m].values()
                    if tab.get(r)
                    for v in tab[r][1].values())
    if not has_value:
        section = label                            # INCOME STATEMENT, CASH FLOW
        continue
    LINES.append((r, section, label, style_for(label)))

print(f"\n{len(LINES)} line items across "
      f"{len(dict.fromkeys(s for _, s, _, _ in LINES))} section(s):")
shown = None
for r, sec, label, _ in LINES:
    if sec != shown:
        print(f"  [{sec}]")
        shown = sec
    print(f"    {LABEL_COLUMN}{r:<3} {label}")

In [ ]:
# ── 4. Build the journey ──────────────────────────────────────────────────────
# One table per tab. Lines run DOWN the page, grouped by section, and each year
# reads left to right as a story: Sep, what changed, Oct, what changed, Nov.
STEPS = list(zip(MONTHS, MONTHS[1:]))          # [(Sep, Oct), (Oct, Nov)]

# the columns inside one year block: a month, with the step before it
COLS = []
for i, m in enumerate(MONTHS):
    if i:
        COLS.append(("step", MONTHS[i - 1], m))
    COLS.append(("value", None, m))

def value_at(tab, row, year, month):
    got = BOOK_DATA.get(month, {}).get(tab, {}).get(row)
    return NAN if got is None else got[1].get(year, NAN)

def step_at(tab, row, year, before, after):
    # (change, how big that move is)
    # Measured against the LARGER of the two figures, not the earlier one. Off a
    # near-zero base the old way reads thousands of percent - a line creeping
    # from 60 to 5,000 outranks one that halved - and a sign change is infinite.
    # This one is bounded, so the ranking means something.
    a = value_at(tab, row, year, before)
    b = value_at(tab, row, year, after)
    if math.isnan(a) or math.isnan(b):
        return NAN, NAN
    d = b - a
    scale = max(abs(a), abs(b))
    return d, (abs(d) / scale if scale else 0.0)

# every step, so the movers can be ranked across the whole book
MOVES = []
for tab in SHEETS:
    for r, section, label, style in LINES:
        for year in YEARS:
            for before, after in STEPS:
                d, rel = step_at(tab, r, year, before, after)
                if not math.isnan(d) and d != 0:
                    MOVES.append(dict(tab=tab, row=r, section=section, label=label,
                                      style=style, year=year, before=before,
                                      after=after, delta=d, rel=rel,
                                      was=value_at(tab, r, year, before),
                                      now=value_at(tab, r, year, after)))
MOVES.sort(key=lambda m: m["rel"], reverse=True)

print(f"{len(SHEETS)} tables x {len(LINES)} lines x "
      f"{len(YEARS)} year(s) x {len(COLS)} column(s)")
print(f"{len(MOVES)} changes between sign-offs; "
      f"{sum(1 for m in MOVES if m['rel'] >= BIG_MOVE)} of them moved "
      f"{BIG_MOVE:.0%} or more")

LAST_STEP = STEPS[-1] if STEPS else None
if LAST_STEP:
    print(f"\nbiggest moves into {LAST_STEP[1]}:")
    for m in [x for x in MOVES if (x["before"], x["after"]) == LAST_STEP][:8]:
        print(f"  {m['rel']:7.1%}  {m['tab']:<30} {m['year']}  {m['label']}"
              f"   {fmt_value(m['was'], m['style']['py'])}"
              f" -> {fmt_value(m['now'], m['style']['py'])}")

In [ ]:
# ── 5. The tables ─────────────────────────────────────────────────────────────
# Change cells are inked by how big the move is against the figure before it -
# intensity only, no colour for good or bad, because what counts as good differs
# line by line. The sign is in the number; the colour says "look here".
from html import escape
from IPython.display import HTML, display

_CSS = """
:root{--bg:#F7F7F5; --panel:#fff; --panel-2:#FAFAFC; --ink:#1F2933;
  --ink-2:#5A6472; --ink-3:#98A1AE; --rule:#E4E6EB; --rule-2:#D3D7DE;
  --accent:#1F3864; --head:#1F3864; --sec:#EEF1F7;}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){
  --bg:#15171C; --panel:#1B1E25; --panel-2:#20242D; --ink:#E8EAEE;
  --ink-2:#AEB6C2; --ink-3:#79828F; --rule:#2A2F39; --rule-2:#39404C;
  --accent:#9BB4E8; --head:#223055; --sec:#242A36;}}
:root[data-theme="dark"]{--bg:#15171C; --panel:#1B1E25; --panel-2:#20242D;
  --ink:#E8EAEE; --ink-2:#AEB6C2; --ink-3:#79828F; --rule:#2A2F39;
  --rule-2:#39404C; --accent:#9BB4E8; --head:#223055; --sec:#242A36;}
*{box-sizing:border-box}
body{margin:0; background:var(--bg); color:var(--ink);
  font:14px/1.5 "IBM Plex Sans",-apple-system,Segoe UI,sans-serif;
  -webkit-font-smoothing:antialiased}
.wrap{max-width:1180px; margin:0 auto; padding:34px 16px 60px}
.mast{border-bottom:1px solid var(--rule); padding-bottom:18px; margin-bottom:24px}
.eyebrow{display:block; font:600 11px/1 "IBM Plex Mono",monospace;
  letter-spacing:.1em; text-transform:uppercase; color:var(--ink-3)}
h1{font:600 30px/1.2 Newsreader,Georgia,serif; margin:10px 0 0}
h2{font:600 17px/1.3 Newsreader,Georgia,serif; margin:0 0 4px}
.dek{margin:0 0 12px; color:var(--ink-2); font-size:13px; max-width:70ch}
.movers, .tabblock{margin-bottom:30px}
.panel{background:var(--panel); border:1px solid var(--rule); border-radius:10px;
  overflow:hidden}
.scroll{overflow:auto}
table{border-collapse:separate; border-spacing:0; width:100%;
  font-variant-numeric:tabular-nums}
th{background:var(--head); color:#fff; font:600 11px/1.25 "IBM Plex Sans",sans-serif;
  text-align:right; padding:6px 10px; white-space:nowrap; position:sticky; top:0; z-index:2}
th.yr{text-align:center; letter-spacing:.05em; text-transform:uppercase}
th.mon{top:30px; background:var(--panel-2); color:var(--ink); font-weight:600}
th.chg{top:30px; background:var(--panel-2); color:var(--ink-3); font-weight:500;
  font-style:italic}
th.corner{left:0; z-index:4; text-align:left; width:270px; vertical-align:bottom}
td{padding:0 10px; height:31px; font-size:12.5px; border-bottom:1px solid var(--rule);
  white-space:nowrap}
td.line{position:sticky; left:0; background:var(--panel); z-index:1; text-align:left;
  font-weight:500; white-space:normal; line-height:1.25; padding:6px 12px;
  border-right:1px solid var(--rule-2); max-width:270px}
td.num{text-align:right; color:var(--ink)}
td.chg{color:var(--ink-2); font-weight:600;
  border-left:1px dotted var(--rule-2)}
td.blank{color:var(--ink-3); font-weight:400; background:transparent!important}
tr.sec td{background:var(--sec); font:600 10px/1 "IBM Plex Mono",monospace;
  letter-spacing:.12em; text-transform:uppercase; color:var(--ink-2); height:26px;
  position:sticky; left:0}
tbody tr:last-child td{border-bottom:none}
table.mv td{border-bottom:1px solid var(--rule)}
table.mv td.pct{width:64px; font-weight:700; text-align:right}
table.mv td.dim{color:var(--ink-3); font-size:11px}
table.mv td.arrow{color:var(--ink-3); width:26px; text-align:center}
.sig{margin-top:18px; display:flex; flex-wrap:wrap; gap:16px;
  font:11px/1.6 "IBM Plex Mono",monospace; color:var(--ink-3)}
.sig code{font:inherit; color:var(--ink-2)}
@media (max-width:720px){.wrap{padding:22px 16px 40px} h1{font-size:23px}
  th.corner, td.line{width:150px; max-width:150px}}
"""

INK = (31, 56, 100)              # the house navy, faded by how small the move is

def ink(rel):
    if rel is None or (isinstance(rel, float) and math.isnan(rel)) or rel <= 0:
        return ""
    t = min(rel / BIG_MOVE, 1.0) ** 0.75
    return "rgba(%d,%d,%d,%.3f)" % (INK + (0.04 + 0.30 * t,))

def pct(x):
    return "" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:+.1%}"

def tables_html(for_page=False):
    sections = list(dict.fromkeys(s for _, s, _, _ in LINES))
    out = []
    for tab in SHEETS:
        out.append('<section class="tabblock"><h2>%s</h2>' % escape(str(tab)))
        out.append('<div class="panel"><div class="scroll"><table><thead>')
        out.append('<tr><th class="corner" rowspan="2" scope="col">Line</th>')
        for y in YEARS:
            out.append('<th class="yr" colspan="%d" scope="colgroup">%s</th>'
                       % (len(COLS), escape(y)))
        out.append('</tr><tr>')
        for y in YEARS:
            for kind, before, after in COLS:
                if kind == "value":
                    out.append('<th class="mon" scope="col">%s</th>' % escape(after))
                else:
                    out.append('<th class="chg" scope="col">%s</th>'
                               % escape("to " + after.split("-")[0]))
        out.append('</tr></thead><tbody>')

        shown = None
        for r, section, label, style in LINES:
            if section != shown:
                out.append('<tr class="sec"><td colspan="%d">%s</td></tr>'
                           % (1 + len(YEARS) * len(COLS), escape(section)))
                shown = section
            out.append('<tr><td class="line">%s</td>' % escape(label))
            for y in YEARS:
                for kind, before, after in COLS:
                    if kind == "value":
                        v = value_at(tab, r, y, after)
                        txt = fmt_value(v, style["py"])
                        out.append('<td class="num%s">%s</td>'
                                   % ("" if txt else " blank", txt or "-"))
                    else:
                        d, rel = step_at(tab, r, y, before, after)
                        if math.isnan(d) or d == 0:
                            out.append('<td class="chg num blank">-</td>')
                        else:
                            sign = "-" if d < 0 else "+"
                            body = style["py"].format(abs(d))
                            out.append('<td class="chg num" style="background:%s" '
                                       'title="%s">%s%s</td>'
                                       % (ink(rel),
                                          escape("%s %s to %s: %s, %s"
                                                 % (y, before, after,
                                                    fmt_value(d, style["py"]),
                                                    "%.0f%% of the larger figure%s"
                                                    % (rel * 100,
                                                       "; changed sign"
                                                       if value_at(tab, r, y, before)
                                                       * value_at(tab, r, y, after) < 0
                                                       else "")), quote=True),
                                          sign, body))
            out.append('</tr>')
        out.append('</tbody></table></div></div></section>')
    return "".join(out)

def movers_html(limit=10):
    if not LAST_STEP:
        return ""
    before, after = LAST_STEP
    picks = [m for m in MOVES if (m["before"], m["after"]) == LAST_STEP][:limit]
    if not picks:
        return ""
    rows = "".join(
        '<tr><td class="num pct" style="background:%s">%s</td>'
        '<td>%s</td><td class="dim">%s</td><td>%s</td>'
        '<td class="num">%s</td><td class="num arrow">to</td><td class="num">%s</td></tr>'
        % (ink(m["rel"]), escape(f"{m['rel']:.0%}"), escape(m["label"]),
           escape(m["year"]), escape(m["tab"]),
           escape(fmt_value(m["was"], m["style"]["py"])),
           escape(fmt_value(m["now"], m["style"]["py"])))
        for m in picks)
    return ('<section class="movers"><h2>What moved into %s</h2>'
            '<p class="dek">The largest changes against %s, by how far each line '
            'moved relative to where it was. Every line is in the tables below.</p>'
            '<div class="panel"><table class="mv"><tbody>%s</tbody></table></div>'
            '</section>' % (escape(after), escape(before), rows))

display(HTML("<style>%s</style>%s%s" % (_CSS, movers_html(), tables_html())))

In [ ]:
# ── 6. Write the workbook ─────────────────────────────────────────────────────
# One sheet per tab, laid out the same way the page is, plus a "Movers" sheet.
def _v(x):
    return None if x is None or (isinstance(x, float) and math.isnan(x)) else float(x)

def _tint(rel):
    # the same fade as the page: stronger ink, bigger move
    if rel is None or (isinstance(rel, float) and math.isnan(rel)) or rel <= 0:
        return None
    t = min(rel / BIG_MOVE, 1.0) ** 0.75
    mix = lambda base: int(round(255 - (255 - base) * (0.10 + 0.55 * t)))
    return PatternFill("solid", fgColor="%02X%02X%02X" % tuple(mix(c) for c in INK))

def write_xlsx(path=None):
    path = OUTPUT_XLSX if path is None else path
    wb = Workbook()
    wb.remove(wb.active)

    head  = PatternFill("solid", fgColor="1F3864")
    sub   = PatternFill("solid", fgColor="D9E1F2")
    secf  = PatternFill("solid", fgColor="EEF1F7")
    hfont = Font(bold=True, color="FFFFFF", size=11)
    sfont = Font(bold=True, color="1F3864", size=10)
    secfn = Font(bold=True, color="5A6472", size=9)
    thin  = Side(style="thin", color="BFBFBF")
    edge  = Border(left=thin, right=thin, top=thin, bottom=thin)
    mid   = Alignment(horizontal="center", vertical="center", wrap_text=True)
    n = len(COLS)

    for tab in SHEETS:
        ws = wb.create_sheet(str(tab)[:31])
        c = ws.cell(1, 1, "Line")
        c.fill, c.font, c.alignment, c.border = head, hfont, mid, edge
        ws.merge_cells(start_row=1, start_column=1, end_row=2, end_column=1)
        ws.cell(2, 1).fill, ws.cell(2, 1).border = head, edge

        col = 2
        for y in YEARS:
            t = ws.cell(1, col, y)
            t.fill, t.font, t.alignment = head, hfont, mid
            ws.merge_cells(start_row=1, start_column=col, end_row=1, end_column=col + n - 1)
            for k, (kind, before, after) in enumerate(COLS):
                ws.cell(1, col + k).border = edge
                s = ws.cell(2, col + k,
                            after if kind == "value" else "to " + after.split("-")[0])
                s.fill, s.font, s.alignment, s.border = sub, sfont, mid, edge
            col += n
        last_col = col - 1

        row = 3
        shown = None
        for r, section, label, style in LINES:
            if section != shown:
                h = ws.cell(row, 1, section)
                h.fill, h.font = secf, secfn
                for k in range(1, last_col + 1):
                    ws.cell(row, k).fill, ws.cell(row, k).border = secf, edge
                shown = section
                row += 1
            t = ws.cell(row, 1, label)
            t.border, t.alignment = edge, Alignment(vertical="center", wrap_text=True)
            col = 2
            for y in YEARS:
                for k, (kind, before, after) in enumerate(COLS):
                    if kind == "value":
                        v = ws.cell(row, col + k, _v(value_at(tab, r, y, after)))
                    else:
                        d, rel = step_at(tab, r, y, before, after)
                        v = ws.cell(row, col + k, _v(d))
                        fill = _tint(rel)
                        if fill:
                            v.fill = fill
                    v.number_format = style["fmt"]
                    v.border = edge
                    v.alignment = Alignment(horizontal="right")
                col += n
            row += 1

        ws.column_dimensions["A"].width = 38
        for k in range(2, last_col + 1):
            ws.column_dimensions[get_column_letter(k)].width = 15
        ws.row_dimensions[2].height = 18
        ws.freeze_panes = "B3"
        ws.sheet_view.showGridLines = False

    if LAST_STEP:
        ws = wb.create_sheet("Movers", 0)
        for k, title in enumerate(("Move", "Line", "Year", "Tab",
                                   LAST_STEP[0], LAST_STEP[1]), start=1):
            c = ws.cell(1, k, title)
            c.fill, c.font, c.alignment, c.border = head, hfont, mid, edge
        row = 2
        for m in [x for x in MOVES if (x["before"], x["after"]) == LAST_STEP]:
            ws.cell(row, 1, m["rel"]).number_format = "0%"
            fill = _tint(m["rel"])
            if fill:
                ws.cell(row, 1).fill = fill
            ws.cell(row, 2, m["label"])
            ws.cell(row, 3, m["year"])
            ws.cell(row, 4, m["tab"])
            ws.cell(row, 5, _v(m["was"])).number_format = m["style"]["fmt"]
            ws.cell(row, 6, _v(m["now"])).number_format = m["style"]["fmt"]
            for k in range(1, 7):
                ws.cell(row, k).border = edge
            row += 1
        for k, w in enumerate((9, 38, 8, 30, 16, 16), start=1):
            ws.column_dimensions[get_column_letter(k)].width = w
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = f"A1:F{row - 1}"
        ws.sheet_view.showGridLines = False

    wb.save(path)
    return os.path.abspath(path)

print("written:", write_xlsx())

In [ ]:
# ── 7. Write the shareable page ───────────────────────────────────────────────
#   signoff_journey.html           - standalone, open it or mail it
#   signoff_journey_artifact.html  - same without the <html>/<body> wrapper,
#                                    which is what the Artifact publisher wants
def build_page(fragment=False):
    books = " ".join('<span>%s: <code>%s</code></span>'
                     % (escape(m), escape(os.path.basename(p)))
                     for m, p in BOOKS if m in MONTHS)
    if RENAMES:
        books += "".join('<span>%s read <code>%s</code> as <code>%s</code></span>'
                         % (escape(month), escape(actual), escape(current))
                         for month, current, actual in RENAMES)

    page = ('<header class="mast">'
            '<span class="eyebrow">%s &middot; %s</span>'
            '<h1>Sign-off Journey</h1>'
            '<p class="dek">The same lines across three sign-offs, read left to right: '
            'a month, what changed, the next month. Change cells are inked by how far '
            'the line moved against the figure before it - the colour only says how '
            'big, never whether it is good. Hover one for the move in full.</p>'
            '</header>\n%s\n%s\n'
            '<div class="sig">%s<span>Generated %s</span></div>'
            % (escape(" / ".join(MONTHS)), escape(STAMP),
               movers_html(), tables_html(for_page=True), books, escape(STAMP)))

    head = ('<meta charset="utf-8">\n<title>Sign-off Journey</title>\n'
            '<link rel="stylesheet" href="https://fonts.googleapis.com/css2?'
            'family=Newsreader:opsz,wght@6..72,400;6..72,600&'
            'family=IBM+Plex+Sans:wght@400;500;600&'
            'family=IBM+Plex+Mono:wght@400;500&display=swap">\n'
            '<style>%s</style>' % _CSS)
    body = '<main class="wrap">%s</main>' % page
    if fragment:
        return head + "\n" + body
    return ('<!doctype html>\n<html lang="en">\n<head>\n'
            '<meta name="viewport" content="width=device-width,initial-scale=1">\n'
            '%s\n</head>\n<body>\n%s\n</body>\n</html>\n' % (head, body))

def write_html(path=None):
    path = OUTPUT_HTML if path is None else path
    alt = os.path.splitext(path)[0] + "_artifact.html"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(build_page(fragment=False))
    with open(alt, "w", encoding="utf-8") as fh:
        fh.write(build_page(fragment=True))
    return os.path.abspath(path), os.path.abspath(alt)

page_path, frag_path = write_html()
print("open this one:    ", page_path)
print("publish this one: ", frag_path)